# Final Global PCA — Defensible Improvements

**CS 663 — Digital Image Processing | MVTec AD**

This is the final Global PCA experiment notebook. It builds on the existing lecture-aligned baseline; it does **not** introduce patch-wise PCA, a neural network, or a different detector.

**Core method unchanged:** fit mean-centred PCA on normal training images, retain the leading principal directions, reconstruct each image, and use the squared reconstruction residual as the anomaly map.

**Controlled refinements evaluated:** (1) retained variance of 90%, 95%, and 98%; (2) optional mild Gaussian smoothing of the residual map; and (3) max versus top-1% residual aggregation for image-level scoring. These are fixed, interpretable ablations. No test-set result is used to select a setting.

**Leakage control:** PCA is fitted on `train/good` only. The deterministic validation split contains normal images only and is used only to set the image-level operating threshold. Official test images/masks are used only for final metrics.

## 1. Setup

Run first with `RUN_ALL_CATEGORIES=False` to smoke-test bottle. After it completes, set it to `True` and run all 15 categories for the report.

In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, label as connected_components
from scipy.integrate import trapezoid
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "global-pca-lecture-review-v2"
if IN_COLAB:
    drive.mount("/content/drive")
    REPO = Path("/content/CS-663")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH,
                        "https://github.com/pranitas-cloud/CS-663.git", str(REPO)], check=True)
else:
    REPO = Path.cwd()
    if not (REPO / "src").is_dir():
        raise RuntimeError("Open this notebook from the CS-663 repository root.")

os.chdir(REPO)
if str(REPO.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO.resolve()))

from src.data.pipelines import load_config, build_dataset, stack_images
from src.pca.global_pca import GlobalPCA

CONFIG = load_config("configs/default.yaml")
DEFAULT_DATA_ROOT = (Path("/content/drive/MyDrive/CS663_Shared/datasets/mvtec_anomaly_detection")
                     if IN_COLAB else REPO / "data" / "mvtec_anomaly_detection")
DATA_ROOT = Path(os.environ.get("MVTEC_DATA_ROOT", DEFAULT_DATA_ROOT)).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = REPO / DATA_ROOT
CONFIG["data"]["root"] = str(DATA_ROOT)
CONFIG["data"]["mask_resize"] = "max"

SEED = int(CONFIG.get("seed", 42))
EXPECTED_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]
RUN_ALL_CATEGORIES = False
SMOKE_TEST_CATEGORY = "bottle"

# Pre-declared ablation settings. Do not select these using test labels.
RETAINED_VARIANCES = [0.90, 0.95, 0.98]
SMOOTHING_SIGMAS = [0.0, 0.75]  # working-resolution pixels (256 x 256)
IMAGE_SCORE_MODES = ["max", "top_1pct"]

np.random.seed(SEED)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"MVTec folder not found: {DATA_ROOT}")

available = {p.name for p in DATA_ROOT.iterdir()
             if p.is_dir() and (p / "train" / "good").is_dir()}
if RUN_ALL_CATEGORIES:
    missing = sorted(set(EXPECTED_CATEGORIES) - available)
    if missing:
        raise FileNotFoundError(f"Missing MVTec categories: {missing}")
    CATEGORIES = EXPECTED_CATEGORIES
else:
    if SMOKE_TEST_CATEGORY not in available:
        raise FileNotFoundError(f"Category not found: {SMOKE_TEST_CATEGORY}")
    CATEGORIES = [SMOKE_TEST_CATEGORY]

print("Repository:", REPO)
print("Branch:", BRANCH)
print("Dataset:", DATA_ROOT)
print("Categories:", CATEGORIES)
print("PCA settings:", RETAINED_VARIANCES)


## 2. Evaluation helpers

The image threshold is the 95th percentile of **normal validation image scores** (nominal 5% validation false-positive rate). AUROC and AP are threshold-independent ranking metrics; precision, recall, and F1 depend on this validation-derived threshold.

AUPRO@30% is approximated using a fixed quantile grid for runtime. Masks use the repository's max-pooling resize rule, which can preserve small defects but slightly expand boundaries at the working resolution.

In [ ]:
def get_maps(dataset, model, sigma):
    labels, masks, maps = [], [], []
    for i in range(len(dataset)):
        item = dataset[i]
        image = item["image"].unsqueeze(0).cpu().numpy()
        amap = model.anomaly_map(image)[0].astype(np.float32)
        if sigma > 0:
            amap = gaussian_filter(amap, sigma=sigma).astype(np.float32)
        mask = item["mask"].squeeze(0).cpu().numpy().astype(bool)
        label_value = int(item["label"])
        if label_value == 1 and not mask.any():
            raise ValueError(f"Anomalous mask became empty after resizing: {item['image_path']}")
        if label_value == 0 and mask.any():
            raise ValueError(f"Normal test image has a non-empty mask: {item['image_path']}")
        labels.append(label_value)
        masks.append(mask)
        maps.append(amap)
    return np.asarray(labels, dtype=np.uint8), np.stack(maps), np.stack(masks)


def image_score(amap, mode):
    flat = np.asarray(amap).reshape(-1)
    if mode == "max":
        return float(flat.max())
    if mode == "top_1pct":
        count = max(1, int(np.ceil(0.01 * flat.size)))
        return float(np.partition(flat, -count)[-count:].mean())
    raise ValueError(f"Unknown image score mode: {mode}")


def compute_aupro(masks, maps, max_fpr=0.30, n_thresholds=128):
    masks = np.asarray(masks, dtype=bool)
    maps = np.asarray(maps, dtype=np.float64)
    if masks.shape != maps.shape or masks.ndim != 3:
        raise ValueError("Expected matching (N,H,W) masks and maps.")
    regions = []
    for mask in masks:
        labels, count = connected_components(mask)
        for region_id in range(1, count + 1):
            region = labels == region_id
            if region.any():
                regions.append(region)
    negatives = int((~masks).sum())
    if not regions or negatives == 0:
        return float("nan")

    thresholds = np.unique(np.quantile(
        maps, np.linspace(1.0, 0.0, n_thresholds)
    ))[::-1]
    fprs, pros = [0.0], [0.0]
    for threshold in thresholds:
        prediction = maps >= threshold
        fpr = np.logical_and(prediction, ~masks).sum() / negatives
        pro = np.mean([
            np.logical_and(prediction, region).sum() / region.sum()
            for region in regions
        ])
        fprs.append(float(fpr))
        pros.append(float(pro))
        if fpr >= max_fpr:
            break

    x, y = np.asarray(fprs), np.asarray(pros)
    order = np.argsort(x, kind="stable")
    x, y = x[order], y[order]
    ux = np.unique(x)
    uy = np.asarray([y[x == value].max() for value in ux])
    if ux[-1] < max_fpr:
        raise RuntimeError("AUPRO curve did not reach the requested FPR.")
    end_y = float(np.interp(max_fpr, ux, uy))
    inside = ux < max_fpr
    curve_x, curve_y = ux[inside], uy[inside]
    if len(curve_x) == 0 or curve_x[0] != 0:
        curve_x = np.insert(curve_x, 0, 0.0)
        curve_y = np.insert(curve_y, 0, 0.0)
    curve_x = np.append(curve_x, max_fpr)
    curve_y = np.append(curve_y, end_y)
    return float(np.clip(trapezoid(curve_y, curve_x) / max_fpr, 0.0, 1.0))


def evaluate_variant(val_maps, test_labels, test_maps, test_masks, score_mode):
    val_scores = np.asarray([image_score(m, score_mode) for m in val_maps])
    threshold = float(np.quantile(val_scores, 0.95, method="higher"))
    test_scores = np.asarray([image_score(m, score_mode) for m in test_maps])
    predictions = (test_scores >= threshold).astype(np.uint8)
    flat_masks = test_masks.reshape(-1).astype(np.uint8)
    flat_scores = test_maps.reshape(-1)
    if np.unique(test_labels).size != 2 or np.unique(flat_masks).size != 2:
        raise ValueError("Test images and masks must each contain both classes.")
    return {
        "image_auroc": float(roc_auc_score(test_labels, test_scores)),
        "image_ap": float(average_precision_score(test_labels, test_scores)),
        "image_precision": float(precision_score(test_labels, predictions, zero_division=0)),
        "image_recall": float(recall_score(test_labels, predictions, zero_division=0)),
        "image_f1": float(f1_score(test_labels, predictions, zero_division=0)),
        "pixel_auroc": float(roc_auc_score(flat_masks, flat_scores)),
        "pixel_ap": float(average_precision_score(flat_masks, flat_scores)),
        "aupro_30pct_fpr": compute_aupro(test_masks, test_maps),
        "validation_threshold": threshold,
    }


# Sanity checks for image scoring and AUPRO implementation.
toy_masks = np.zeros((2, 16, 16), dtype=bool)
toy_masks[0, 5:10, 5:10] = True
toy_maps = np.zeros((2, 16, 16), dtype=np.float32)
toy_maps[toy_masks] = 1.0
assert np.isclose(compute_aupro(toy_masks, toy_maps), 1.0)
assert image_score(toy_maps[0], "max") == 1.0
assert np.isclose(image_score(toy_maps[0], "top_1pct"), 1.0)
print("Evaluation helper sanity checks passed.")


## 3. Fit once per category; compare controlled variants

The full ordered PCA basis is fitted once per category using the repository's GlobalPCA implementation and the sample-space Gram/SVD route. Each retained-variance setting is then obtained by truncating that same ordered basis. This makes the variance comparison controlled and avoids three independent fits.

For each setting, the notebook evaluates raw versus smoothed residual maps and max versus top-1% image scoring. Smoothing changes the localization map as well as image scoring; the image-score aggregation changes only the image-level score. Results are reported as ablations, not as a promise that every change improves performance.

In [ ]:
all_rows = []
preview = None

for category in CATEGORIES:
    print(f"\\n===== {category} =====")
    start = time.perf_counter()
    train = build_dataset(CONFIG, "pca", "train", category)
    val = build_dataset(CONFIG, "pca", "val", category)
    test = build_dataset(CONFIG, "pca", "test", category)

    if min(len(train), len(val), len(test)) == 0:
        raise ValueError(f"An empty split was found for {category}.")
    if not all(sample[2] == "good" for sample in train.samples):
        raise ValueError("PCA training split must contain train/good only.")
    if not all(sample[2] == "good" for sample in val.samples):
        raise ValueError("Validation split must contain normal images only.")

    # Fit the full available basis once; all variants share this fitted basis.
    full_model = GlobalPCA(
        n_components="all", method="svd", residual="squared", ddof=0
    ).fit(stack_images(train).cpu().numpy())

    for variance in RETAINED_VARIANCES:
        k = int(np.searchsorted(
            np.cumsum(full_model.eigenvalues_all_) / full_model.total_variance_,
            variance
        ) + 1)
        model = full_model.truncate(k)
        actual_variance = float(model.explained_variance_ratio_.sum())

        val_labels_raw, val_maps_raw, _ = get_maps(val, model, sigma=0.0)
        test_labels_raw, test_maps_raw, test_masks = get_maps(test, model, sigma=0.0)
        if np.any(val_labels_raw != 0):
            raise ValueError("Validation set unexpectedly contains anomalous images.")

        for sigma in SMOOTHING_SIGMAS:
            if sigma == 0:
                val_maps, test_maps = val_maps_raw, test_maps_raw
            else:
                val_maps = gaussian_filter(
                    val_maps_raw, sigma=(0, sigma, sigma)
                ).astype(np.float32)
                test_maps = gaussian_filter(
                    test_maps_raw, sigma=(0, sigma, sigma)
                ).astype(np.float32)

            for score_mode in IMAGE_SCORE_MODES:
                metrics = evaluate_variant(
                    val_maps, test_labels_raw, test_maps, test_masks, score_mode
                )
                all_rows.append({
                    "category": category,
                    "retained_variance_target": variance,
                    "retained_variance_actual": actual_variance,
                    "components": k,
                    "smoothing_sigma": sigma,
                    "image_score": score_mode,
                    "training_images": len(train),
                    "validation_images": len(val),
                    "test_images": len(test),
                    **metrics
                })

        if preview is None and np.any(test_labels_raw == 1):
            idx = int(np.flatnonzero(test_labels_raw == 1)[0])
            preview = (
                test[idx]["image"].squeeze().cpu().numpy(),
                test_maps_raw[idx],
                gaussian_filter(test_maps_raw[idx], sigma=0.75),
                test_masks[idx]
            )

    print(f"Completed {category} in {time.perf_counter() - start:.1f}s")

results = pd.DataFrame(all_rows)
metric_columns = [
    "image_auroc", "image_ap", "image_precision", "image_recall",
    "image_f1", "pixel_auroc", "pixel_ap", "aupro_30pct_fpr"
]
summary = (
    results.groupby(
        ["retained_variance_target", "smoothing_sigma", "image_score"]
    )[metric_columns]
    .mean()
    .reset_index()
)
print("Category-level results:")
display(results.round(4))
print("Macro averages for the categories included in this run:")
display(summary.round(4))

out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)
category_path = out_dir / "global_pca_final_categories.csv"
summary_path = out_dir / "global_pca_final_macro.csv"
results.to_csv(category_path, index=False)
summary.to_csv(summary_path, index=False)
print("Saved:", category_path)
print("Saved:", summary_path)

if preview is not None:
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    axes[0].imshow(preview[0], cmap="gray")
    axes[0].set_title("Test image")
    axes[1].imshow(preview[1], cmap="inferno")
    axes[1].set_title("Raw PCA residual")
    axes[2].imshow(preview[2], cmap="inferno")
    axes[2].set_title("Smoothed residual")
    axes[3].imshow(preview[3], cmap="gray")
    axes[3].set_title("Ground-truth mask")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 4. How to report and defend this

- **PCA core:** exactly the lecture idea—mean-centre normal images, obtain principal directions, retain leading directions, project and reconstruct. The residual is the project-specific anomaly signal.
- **90/95/98% retained variance:** a dimensionality ablation. More components improve normal reconstruction but may also reconstruct defect structure. No setting is universally best.
- **Gaussian smoothing (sigma 0.75):** a mild spatial denoising step on the residual map. It can suppress isolated pixel noise and make regions more coherent, but can blur tiny defects. The raw map remains in the comparison.
- **Top-1% mean:** averages the strongest 1% of residual pixels, reducing dependence on one extreme pixel compared with max. It may dilute very small defects; hence both scores are reported.
- **Threshold:** the 95th percentile of normal validation image scores; it is not tuned on test labels.
- **Metrics:** image/pixel AUROC and AP assess ranking; image precision/recall/F1 depend on the validation threshold; AUPRO evaluates region overlap under a pixel false-positive constraint.

**Important:** run all 15 categories before reporting a dataset macro. Do not claim an improvement unless the executed results support it. If a refinement helps image AUROC but hurts pixel AUROC/AUPRO, report that trade-off. Global PCA remains position-sensitive and may struggle with local texture changes, misalignment, and defects that resemble normal global variation.

**Reference basis:** PCA formulation follows the CS663 PCA lecture. Residual smoothing and top-k aggregation are simple image-processing/scoring choices tested as ablations; they are not claimed as novel methods or as guaranteed improvements.